In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import os
from pathlib import Path

In [ ]:
CACHE_ROOT = "/cs/labs/oabend/tomer.shahaf/hf_cache_root"
research_df_tmp_with_cosines_parquet_path = os.path.join(CACHE_ROOT, "df_sampled_100k_tmp_with_cosines.pqt")
research_high_semantic_conversations_parquet_path = os.path.join(CACHE_ROOT, "research_high_semantic_conversations.pqt")
research_high_semantic_conversations_parquet_output_path = os.path.join(CACHE_ROOT, "research_high_semantic_conversations_parquet_output.pqt")
final_chunks_folder = os.path.join(CACHE_ROOT, "final_chunks")
final_conversations_df_parquet_path = os.path.join(CACHE_ROOT, "final_conversations_df_parquet.pqt")

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
# conversations_df = pl.read_parquet(research_high_semantic_conversations_parquet_output_path)
# conversations_df.shape

In [ ]:
conversations_df = pl.read_parquet(final_conversations_df_parquet_path)
# conversations_df.write_parquet(final_conversations_df_parquet_path)
conversations_df.shape

In [ ]:
parquet_files = list(Path(final_chunks_folder).glob("*.pqt"))

dfs_list = []
for file in tqdm(parquet_files):
    dfs_list.append(pl.read_parquet(file))

conversations_df = pl.concat(dfs_list)
conversations_df.shape

In [ ]:
conversations_df.columns

In [ ]:
px.histogram(conversations_df["count_before_model_semantic_change"])

In [ ]:
px.histogram(conversations_df["model_name"])

In [ ]:
conversations_df["user_prompts"].sample()[0]

In [ ]:
# conversations_df_copy = conversations_df
# conversations_df = conversations_df[:50_000]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
MAX_FEATURES_IN_TFIDF = 10_000

In [ ]:
user_prompts_clean_df = (
    conversations_df[:50_000].lazy()  
    .with_row_index(name="conv_id") 
    .explode("user_prompts")
    .with_columns(
        pl.col("user_prompts").cum_count().over("conv_id").alias("turn_index")
    )
    .filter(pl.col("turn_index") < 5)
    .group_by("turn_index")
    .agg(pl.col("user_prompts")) 
    .collect() 
)
user_prompts_clean_df = user_prompts_clean_df.sort("turn_index")

In [ ]:
# model_answers_clean_df = (
#     conversations_df.lazy()  
#     .with_row_index(name="conv_id") 
#     .explode("model_answers")
#     .with_columns(
#         pl.col("model_answers").cum_count().over("conv_id").alias("turn_index")
#     )
    
#     .filter(pl.col("turn_index") < 5)
#     .group_by("turn_index")
#     .agg(pl.col("model_answers")) 
#     .collect() 
# )
# model_answers_clean_df = model_answers_clean_df.sort("turn_index")

In [ ]:
user_prompts_turns_corpus = (
    user_prompts_clean_df
    .sort("turn_index")
    .select(pl.col("user_prompts").list.join(" "))
    .to_series()
    .to_list()
)

In [ ]:
# model_answers_turns_corpus = (
#     model_answers_clean_df
#     .sort("turn_index")
#     .select(pl.col("model_answers").list.join(" "))
#     .to_series()
#     .to_list()
# )

In [ ]:
import time
from tqdm import tqdm

t0 = time.time()

with tqdm(total=4, desc="Processing", unit="step") as pbar:
    pbar.set_description("Vectorizing corpus")
    vec = CountVectorizer(stop_words='english', max_features=MAX_FEATURES_IN_TFIDF)
    X = vec.fit_transform(user_prompts_turns_corpus)
    tqdm.write(f"Vectorization done: {time.time()-t0:.2f}s")
    pbar.update(1)
    
    pbar.set_description("Creating dataframe")
    df_counts = pd.DataFrame(
        X.toarray(), 
        columns=vec.get_feature_names_out(),
        index=[f"Turn {i}" for i in sorted(user_prompts_clean_df["turn_index"])]
    )
    tqdm.write(f"DataFrame created: {time.time()-t0:.2f}s")
    pbar.update(1)
    
    pbar.set_description("Normalizing probabilities")
    df_probs = df_counts.div(df_counts.sum(axis=1), axis=0)
    tqdm.write(f"Probabilities calculated: {time.time()-t0:.2f}s")
    pbar.update(1)
    
    pbar.set_description("Computing differences")
    df_final = df_probs.T
    df_final['Diff (T4 - T1)'] = df_final['Turn 4'] - df_final['Turn 1']
    tqdm.write(f"Analysis complete: {time.time()-t0:.2f}s")
    pbar.update(1)

print("\n--- Words that DROP the most (Openers) ---")
print(df_final.sort_values('Diff (T4 - T1)').head(10))
print("\n--- Words that RISE the most (Follow-ups) ---")
print(df_final.sort_values('Diff (T4 - T1)', ascending=False).head(10))

In [ ]:
drop_df = df_final.sort_values('Diff (T4 - T1)').head(10).reset_index().rename(columns={'index': 'Word'})
rise_df = df_final.sort_values('Diff (T4 - T1)', ascending=False).head(10).reset_index().rename(columns={'index': 'Word'})

In [ ]:
# show drop_df

import time
t0 = time.time()

# 1. Setup: Get the list of words we are currently plotting
words_to_plot = drop_df['Word'].tolist()
print(time.time()-t0)

# 2. Get the specific data slices
# A. Probabilities (You already have this)
probs_subset = df_final.loc[words_to_plot, ['Turn 1', 'Turn 2', 'Turn 3', 'Turn 4']]

# B. Absolute Counts (Retrieve from the df_counts we created earlier)
# We transpose (.T) so rows are Words and columns are Turns, matching probs_subset
counts_subset = df_counts[words_to_plot].T 
print(time.time()-t0)

# C. Multiplication Factors (Current Prob / Turn 1 Prob)
# If a word drops from 0.10 to 0.05, Factor is 0.5x
factors_subset = probs_subset.div(probs_subset['Turn 1'], axis=0)

# 3. Melt all three individually
# We use reset_index() so 'Word' becomes a column
melt_probs = probs_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Probability').rename(columns={'index': 'Word'})
melt_counts = counts_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Count').rename(columns={'index': 'Word'})
melt_factor = factors_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Factor').rename(columns={'index': 'Word'})

# 4. Merge them into one "Rich" DataFrame
df_rich = melt_probs.merge(melt_counts, on=['Word', 'Turn'])\
                    .merge(melt_factor, on=['Word', 'Turn'])
print(time.time()-t0)

# Preview to ensure it looks right
print(df_rich.head())

In [ ]:
fig = px.line(
    df_rich, 
    x="Turn", 
    y="Probability", 
    color="Word", 
    markers=True,
    title="Evolution of User Language - drop words",
    template="plotly_white",
    height=500,
    # --- CUSTOM HOVER CONFIGURATION ---
    hover_data={
        "Turn": False,             # Hide (already on x-axis)
        "Word": False,             # Hide (already in legend)
        "Probability": ":.4f",     # Format: 4 decimal places
        "Count": True,             # Show raw count
        "Factor": ":.2f"           # Format: 2 decimal places
    }
)

# Optional: Rename the labels in the tooltip to be friendlier
fig.update_traces(
    hovertemplate="<br>".join([
        "<b>%{x}</b>",
        "Prob: %{y:.4f}",
        "Count: %{customdata[1]}",    # Accesses 'Count'
        "Growth: %{customdata[2]:.2f}x" # Accesses 'Factor'
    ])
)

fig.update_layout(yaxis_title="Probability (Frequency)", hovermode="x unified")
fig.show()

In [ ]:
# show rise_df

# 1. Setup: Get the list of words we are currently plotting
words_to_plot = rise_df['Word'].tolist()

# 2. Get the specific data slices
# A. Probabilities (You already have this)
probs_subset = df_final.loc[words_to_plot, ['Turn 1', 'Turn 2', 'Turn 3', 'Turn 4']]

# B. Absolute Counts (Retrieve from the df_counts we created earlier)
# We transpose (.T) so rows are Words and columns are Turns, matching probs_subset
counts_subset = df_counts[words_to_plot].T 

# C. Multiplication Factors (Current Prob / Turn 1 Prob)
# If a word drops from 0.10 to 0.05, Factor is 0.5x
factors_subset = probs_subset.div(probs_subset['Turn 1'], axis=0)

# 3. Melt all three individually
# We use reset_index() so 'Word' becomes a column
melt_probs = probs_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Probability').rename(columns={'index': 'Word'})
melt_counts = counts_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Count').rename(columns={'index': 'Word'})
melt_factor = factors_subset.reset_index().melt(id_vars='index', var_name='Turn', value_name='Factor').rename(columns={'index': 'Word'})

# 4. Merge them into one "Rich" DataFrame
df_rich = melt_probs.merge(melt_counts, on=['Word', 'Turn'])\
                    .merge(melt_factor, on=['Word', 'Turn'])

# Preview to ensure it looks right
print(df_rich.head())

In [ ]:
fig = px.line(
    df_rich, 
    x="Turn", 
    y="Probability", 
    color="Word", 
    markers=True,
    title="Evolution of User Language - rise words",
    template="plotly_white",
    height=500,
    # --- CUSTOM HOVER CONFIGURATION ---
    hover_data={
        "Turn": False,             # Hide (already on x-axis)
        "Word": False,             # Hide (already in legend)
        "Probability": ":.4f",     # Format: 4 decimal places
        "Count": True,             # Show raw count
        "Factor": ":.2f"           # Format: 2 decimal places
    }
)

# Optional: Rename the labels in the tooltip to be friendlier
fig.update_traces(
    hovertemplate="<br>".join([
        "<b>%{x}</b>",
        "Prob: %{y:.4f}",
        "Count: %{customdata[1]}",    # Accesses 'Count'
        "Growth: %{customdata[2]:.2f}x" # Accesses 'Factor'
    ])
)

fig.update_layout(yaxis_title="Probability (Frequency)", hovermode="x unified")
fig.show()

In [ ]:
### vectors ### 

In [ ]:
query_all_embeddings_df_path = os.path.join(CACHE_ROOT, "query_all_embeddings.pqt")

In [ ]:
query_all_embeddings_df = pl.read_parquet(query_all_embeddings_df_path)

In [ ]:
query_all_embeddings_df.head()

In [ ]:
import numpy as np
import numpy as np

# 1. Extract the first row's value directly as a Python list
# .item(0) is instant and avoids overhead
embedding_data = query_all_embeddings_df["embeddings"].item(0)

# 2. Inspect the structure
print(f"Type: {type(embedding_data)}")  # Should be <class 'list'>
print(f"Outer list length: {len(embedding_data)}") 
print(f"First vector sample: {embedding_data[0]}")

# 3. Convert to NumPy
# Since it is a list of lists, this creates a 2D Matrix
emb_array = np.array(embedding_data)

print(f"\nNumPy Array Shape: {emb_array.shape}")
# Likely (n_tokens, hidden_dim) or similar
# Assuming 'embedding_data' is the variable you inspected above
# 1. Iterate through the container and convert each inner Series to a numpy array
vectors_list = [v.to_numpy() for v in embedding_data]

# 2. Stack them into a 2D matrix
matrix = np.vstack(vectors_list)

print(f"Final Shape: {matrix.shape}") 
# Expected Output: (2, 768) 
# (2 vectors, each with 768 dimensions)